# 이미지 생성 모델 개요: VAE, GAN, Diffusion

**생성 모델**은 학습 데이터가 가진 패턴과 분포를 익혀 기존 데이터를 복사하지 않고 새로운 데이터를 만드는 모델이다. 이미지 생성에서는 새로운 샘플을 만들거나, 조건에 맞는 이미지를 생성하거나, 입력 이미지를 변형하기 위해 사용한다.

VAE, GAN, Diffusion은 모두 이미지를 생성하지만 학습 목표와 생성 과정이 다르다. 이번 수업에서는 세 모델을 `입력 → 변환 → 출력` 흐름으로 비교하고, 다음 GAN과 Diffusion 실습에서 어떤 부분을 코드로 확인할지 연결한다.


## 세 모델의 생성 경로 비교

![VAE, GAN, Diffusion 생성 경로 비교](https://cdn.jsdelivr.net/gh/goat-skn-ai/image-repo@20132d4eb888d7c341e15c2fb4573d0ecf60c0da/09_multimodal/05_image_generation/01_model_family_comparison.svg)

VAE는 인코더와 디코더 사이의 잠재변수, GAN은 생성자와 판별자의 경쟁, Diffusion은 노이즈 추가와 반복 제거가 핵심이다. 아래 설명에서는 그림의 각 화살표를 대표 수식과 연결한다.


## VAE: 잠재분포를 학습하는 생성 모델

**VAE(Variational Autoencoder)** 는 입력을 하나의 고정 벡터가 아니라 잠재변수의 확률분포로 압축하고, 그 분포에서 샘플링한 값을 디코딩해 이미지를 생성한다.

잠재공간을 연속적이고 규칙적으로 만들기 때문에 이미지 보간, 변형, 표현 학습에 활용한다.

![잠재공간을 이동할 때 생성된 얼굴이 연속적으로 변하는 모습](https://miro.medium.com/v2/resize:fit:1280/0*dwtvGrRWRAUJuZm4.gif)

얼굴이 갑자기 전혀 다른 모습으로 바뀌지 않고 조금씩 이어서 변하는 과정에 주목한다.<br>
이는 잠재변수 $z$의 값을 연속적으로 이동하면 decoder가 만든 결과도 연속적으로 변할 수 있다는 잠재공간의 직관을 보여준다. <br>

- **입력**: 학습 이미지 $x$
- **변환**: 인코더 $q_\phi(z|x)$가 잠재분포를 만들고, 그 분포에서 $z$를 샘플링한다.
- **출력**: 디코더 $p_\theta(x|z)$가 $z$로부터 입력과 유사한 이미지 또는 새로운 이미지를 만든다.

---

VAE는 다음 ELBO(Evidence Lower Bound)를 크게 만드는 방향으로 학습한다.

$$
\mathcal{L}_{\text{VAE}}
= \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]
- D_{KL}\bigl(q_\phi(z|x)\|p(z)\bigr)
$$

- $x$는 입력 이미지이고 $z$는 압축된 잠재변수이다.
- $q_\phi(z|x)$는 인코더가 만든 잠재분포이고 $p_\theta(x|z)$는 디코더의 복원분포이다.
- 첫 번째 항은 입력 정보를 잘 복원하도록 만들고, $D_{KL}$ 항은 잠재분포를 기준분포 $p(z)$에 가깝게 정렬한다.

**장점**은 잠재공간을 탐색하고 보간하기 쉽다는 점이다. **한계**는 재구성과 분포 정렬을 함께 최적화하므로 생성 결과가 상대적으로 부드럽거나 흐릿해질 수 있다는 점이다. 이미지 표현을 압축하거나 잠재공간을 조작해야 할 때 적합하며, Latent Diffusion에서는 픽셀 이미지를 작은 잠재표현으로 바꾸는 구성 요소로도 사용한다.


## GAN: 생성자와 판별자가 경쟁하는 생성 모델

**GAN(Generative Adversarial Network)** 은 가짜 샘플을 만드는 생성자 $G$와 실제·가짜 샘플을 구분하는 판별자 $D$를 번갈아 학습한다. <br>
생성자는 판별자를 속일 만큼 실제와 비슷한 샘플을 만들고, 판별자는 두 분포를 더 정확하게 구분하는 방향으로 갱신된다.

![두 얼굴의 특징을 결합해 새로운 얼굴을 생성하는 GAN 응용](https://miro.medium.com/v2/resize:fit:1040/0*x2RTBmZUtevU0lW7.gif)

위쪽의 `Source A`는 성별, 나이, 머리 길이, 안경, 자세와 같은 특징을 제공하고, 왼쪽의 `Source B`는 나머지 얼굴 특징을 제공한다. <br>
교차 지점의 이미지는 두 입력의 특징을 결합한 결과이며, GAN의 잠재표현을 조절하면 생성 이미지의 속성을 나누어 바꿀 수 있음을 보여준다.

![GAN의 반복 학습에 따라 생성 결과가 개선되는 과정](https://www.tensorflow.org/tutorials/generative/images/gan2.png)

처음에는 생성 결과가 실제 데이터와 크게 달라 판별자가 쉽게 가짜로 구분한다. 생성자는 판별 결과에서 전달된 gradient를 이용해 출력을 고치며, 이 과정을 반복하면 실제 데이터와 비슷한 특징을 만들게 된다. 마지막의 초록색 표시는 정답 이미지를 복사했다는 뜻이 아니라, 판별자가 생성 결과를 실제로 판단할 정도로 구분이 어려워졌다는 뜻이다.

---

- **입력**: 기준분포에서 뽑은 noise 벡터 $z$와 학습 데이터의 실제 이미지 $x$
- **변환**: $G(z)$가 가짜 이미지를 만들고, $D$가 $x$와 $G(z)$를 각각 평가한다.
- **출력**: 학습이 진행되면 생성자는 실제 데이터 분포와 비슷한 새로운 이미지를 만든다.

대표적인 GAN 목적함수는 다음과 같다.

$$
\min_G \max_D V(D,G)
= \mathbb{E}_{x\sim p_{data}}[\log D(x)]
+ \mathbb{E}_{z\sim p_z}[\log(1-D(G(z)))]
$$

- $p_{data}$는 실제 데이터 분포이고 $p_z$는 noise를 뽑는 기준분포이다.
- $D(x)$는 실제 샘플을 실제로 판단하도록, $D(G(z))$는 생성 샘플을 구분하도록 학습한다.
- 이론식의 $D(\cdot)$는 0과 1 사이의 값으로 표현하지만, 다음 실습은 수치 안정성을 위해 **logit**과 `BCEWithLogitsLoss`를 사용한다.

**장점**은 한 번의 생성자 forward로 빠르고 선명한 샘플을 만들 수 있다는 점이다. **한계**는 두 모델의 균형을 맞추기 어렵고, 다양한 입력이 거의 같은 결과로 모이는 **mode collapse**가 발생할 수 있다는 점이다. 빠른 추론이나 특정 도메인의 고품질 생성이 중요할 때 후보가 된다.


## Diffusion: 노이즈를 예측하며 이미지를 복원하는 생성 모델

**Diffusion 모델**은 학습 이미지에 단계적으로 noise를 섞는 정방향 과정을 정의하고, 각 단계에서 섞인 noise를 예측하는 denoiser를 학습한다.<br>
 생성할 때는 무작위 noise에서 시작해 예측된 noise를 여러 번 제거하므로 텍스트 조건과 결합한 고품질 이미지 생성에 널리 사용한다.

![무작위 noise에서 이미지를 복원하는 Diffusion 반복 과정](https://learnopencv.com/wp-content/uploads/2023/01/diffusion-models-unconditional_image_generation.gif)

왼쪽의 무작위 noise가 모델과 scheduler의 반복 단계를 통과하면서 오른쪽의 이미지로 바뀌는 과정이다. 이는 학습 이미지에 noise를 추가하는 정방향 과정이 아니라, 생성 시 noise를 반복해서 제거하는 **역방향 과정**을 나타낸다.

---

- **입력**: 학습 시 원본 이미지 $x_0$, 생성 시 무작위 noise $x_T$
- **변환**: 학습 시 임의의 단계 $t$에서 noise를 예측하고, 생성 시 scheduler가 denoiser의 예측으로 $x_t$를 $x_{t-1}$로 반복 갱신한다.
- **출력**: 반복 제거가 끝난 생성 이미지 $x_0$

원본 이미지에서 임의 단계의 noisy image를 직접 만드는 식은 다음과 같다.

$$
x_t = \sqrt{\bar{\alpha}_t}\,x_0
+ \sqrt{1-\bar{\alpha}_t}\,\epsilon,
\qquad \epsilon\sim\mathcal{N}(0,I)
$$

- $t$는 noise 단계이고 $\bar{\alpha}_t$는 해당 단계까지 남아 있는 원본 신호의 누적 비율이다.
- $\epsilon$은 정규분포에서 뽑은 noise이며, $t$가 커질수록 $x_t$에서 원본 신호가 줄어든다.
- 실제 생성에서는 정답 noise를 알 수 없으므로 denoiser가 $\epsilon$ 또는 이에 대응하는 값을 예측한다.

**장점**은 학습이 비교적 안정적이고 품질·다양성·조건 제어가 우수하다는 점이다. **한계**는 여러 denoising step이 필요해 생성 시간과 연산 비용이 커질 수 있다는 점이다. 텍스트 프롬프트 제어와 범용 이미지 생성이 중요할 때 우선 검토한다.


## 세 모델의 선택 경계

| 비교 기준 | VAE | GAN | Diffusion |
|---|---|---|---|
| 핵심 학습 구조 | 인코더와 디코더, 잠재분포 정렬 | 생성자와 판별자의 경쟁 | noise 추가와 noise 예측 |
| 생성 입력 | 잠재변수 $z$ | noise $z$ | 초기 noise $x_T$와 선택적 조건 |
| 대표 강점 | 구조화된 잠재공간과 보간 | 빠른 추론과 선명한 결과 | 안정적인 학습과 강한 조건 제어 |
| 대표 한계 | 흐릿한 복원이 생길 수 있음 | 학습 불안정과 mode collapse | 반복 추론에 따른 시간과 비용 |
| 우선 사용 위치 | 표현 학습, 압축, 잠재공간 조작 | 빠른 특정 도메인 생성 | 텍스트 기반 범용 이미지 생성 |

모델은 이름보다 목적과 제약을 기준으로 선택한다.

- 잠재공간을 해석하거나 연속적으로 조작하려면 VAE를 먼저 검토한다.
- 추론 속도와 특정 도메인의 선명한 출력이 중요하면 GAN을 검토한다.
- 프롬프트 제어, 범용성, 결과 다양성이 중요하면 Diffusion을 검토한다.
- 실제 서비스에서는 데이터 권리, GPU 메모리, 생성 지연 시간, API 비용과 안전 정책도 함께 확인한다.

다음 노트북에서는 GAN의 판별자와 생성자가 한 번씩 갱신되는 학습 iteration을 코드로 확인한다. 이어지는 Diffusion 노트북에서는 정방향 noise 식과 실제 이미지 생성 pipeline을 연결한다.
